# Outbound Freight Allocation Optimization

## Problem Statement

The Scenario: The Outbound Freight Allocation Problem
A regional distribution center is planning its weekly outbound freight allocation. To move palletized merchandise to local retail stores, the transportation team utilizes three different fleet types: an Internal Fleet, a Dedicated Contract Carrier, and an Expedited Carrier.

Each fleet charges a different base rate per trailer load, and each possesses a unique historical "On-Time Rating" and a "Fuel Efficiency Score."

The Goal
The transportation team needs to minimize the total operational cost of moving all required freight out of the distribution center.

The Decisions to Make
You need to determine exactly how many trailer loads to allocate to each of the three fleets for the week.

The Business Rules (Constraints)
The allocation plan must strictly follow these rules to ensure service level agreements are met and contracts are honored:

Total Volume: Exactly 800 trailer loads must be moved this week to clear the distribution center's yard and meet store demand.

Contractual Proportions: To satisfy the annual minimum volume guarantee, the Dedicated Contract Carrier must handle at least 30% of the total weekly loads.

Equipment Capacity: Due to a shortage of available drivers and tractors, the Internal Fleet can handle an absolute maximum of 250 loads this week.

Service Standard (On-Time): The final blended delivery schedule must have an average On-Time Rating of at least 96% to meet strict store delivery expectations. (The three fleets all have different historical on-time ratings that will average out based on the volume assigned to each).

Sustainability Standard (Efficiency): The final fleet allocation must have an average Fuel Efficiency Score of at least 7.0 to remain compliant with corporate sustainability targets.

Physical Reality: We cannot assign negative trailer loads to any fleet.


In [1]:
import pandas as pd
import numpy as np
from autograd import grad, jacobian
import autograd.numpy as anp
from scipy.optimize import minimize
from docplex.mp.model import Model

In [2]:
data = pd.DataFrame(
    columns=[
        "Fleet_Type", 
        "Variable", 
        "Cost_Per_Load", 
        "On_Time_Rating", 
        "Efficiency_Score"
    ], 
    data=[
        ["Internal Fleet", "x1", 600, 98.0, 8.0], 
        ["Dedicated Contract", "x2", 850, 94.0, 7.0], 
        ["Expedited Carrier", "x3", 1200, 99.5, 6.0]
    ]
)

In [3]:
data

,Fleet_Type,Variable,Cost_Per_Load,On_Time_Rating,Efficiency_Score
0,Internal Fleet,x1,600,98.0,8.0
1,Dedicated Contract,x2,850,94.0,7.0
2,Expedited Carrier,x3,1200,99.5,6.0


In [4]:
total_load_target = 800
internal_fleet_load_max = 250
on_time_rating_min = 96
efficiency_score_min = 7

minimize cost
cost = cost_rate_1 * x1 + cost_rate_2 * x2 + cost_rate_3 * x3  
x1 = loads from internal fleet  
x2 = loads from dedicated contract carrier  
x3 = loads from expedited carrier  

theta = [x1, x2, x3]  

constraints:  
g1(theta) - equality: x1 + x2 + x3 - total_load_target= 0  
g2(theta) - inequality: x2 - 0.3*(x1 + x2 + x3) >= 0  
g3(theta) - inequality: (ot_rating1 * x1 + ot_rating2 * x2 + ot_rating3 * x3) / (x1 + x2 + x3) - on_time_rating_min >= 0  
g4(theta) - inequality: (eff1 * x1 + eff2 * x2 + eff3 * x3) / (x1 + x2 + x3) - efficiency_score_min >= 0  

bounds:  
bound1: 0 <= x1 <= internal_load_fleet_max  
bound2: 0 <= x2  
bound3: 0 <= x3

In [5]:
### Creating relevant variables
cost_rate_1 = data[data["Variable"] == "x1"]["Cost_Per_Load"].values[0]
cost_rate_2 = data[data["Variable"] == "x2"]["Cost_Per_Load"].values[0]
cost_rate_3 = data[data["Variable"] == "x3"]["Cost_Per_Load"].values[0]

ot_rating_1 = data[data["Variable"] == "x1"]["On_Time_Rating"].values[0]
ot_rating_2 = data[data["Variable"] == "x2"]["On_Time_Rating"].values[0]
ot_rating_3 = data[data["Variable"] == "x3"]["On_Time_Rating"].values[0]

eff_score_1 = data[data["Variable"] == "x1"]["Efficiency_Score"].values[0]
eff_score_2 = data[data["Variable"] == "x2"]["Efficiency_Score"].values[0]
eff_score_3 = data[data["Variable"] == "x3"]["Efficiency_Score"].values[0]

In [6]:
def objective(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return cost_rate_1 * x1 + cost_rate_2 * x2 + cost_rate_3 * x3

In [7]:
def constraint1(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return x1 + x2 + x3 - total_load_target

In [8]:
def constraint2(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return 0.7*x2 - 0.3*x1 - 0.3*x3

In [9]:
def constraint3(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return ((ot_rating_1 - 96) * x1 + (ot_rating_2 - 96) * x2 + (ot_rating_3 - 96) * x3)

In [10]:
def constraint4(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return ((eff_score_1 - efficiency_score_min) * x1 + (eff_score_2 - efficiency_score_min) * x2 + (eff_score_3 - efficiency_score_min) * x3)

In [11]:
bound1 = np.array([0, internal_fleet_load_max])
bound2 = np.array([0,None])
bound3 = np.array([0,None])
bounds = np.array([bound1, bound2, bound3])

In [12]:
theta0 = np.array([150,350,300])

In [13]:
constraints1 = [
    {"type":"eq", "fun":constraint1},
    {"type":"ineq", "fun":constraint2},
    {"type":"ineq", "fun":constraint3},
    {"type":"ineq", "fun":constraint4}
]


## Method 1: Using SciPy and not passing in gradient or constraint jacobians

In [14]:
sol1 = minimize(fun = objective, x0 = theta0, method = "SLSQP", bounds = bounds, constraints = constraints1)

In [15]:
sol1.x

array([250.        , 440.90909091, 109.09090909])

In [16]:
sol1.fun

np.float64(655681.8181781151)

In [17]:
sol1

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 655681.8181781151
           x: [ 2.500e+02  4.409e+02  1.091e+02]
         nit: 3
         jac: [ 6.000e+02  8.500e+02  1.200e+03]
        nfev: 9
        njev: 2
 multipliers: [ 9.773e+02  0.000e+00  6.364e+01  0.000e+00]

## Method 2: Using SciPy and calculating gradient and jacobians myself using calculus

In [18]:
def objective_gradient(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return anp.array([cost_rate_1, cost_rate_2, cost_rate_3], dtype = float)

In [19]:
def constraint1_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return anp.array([1,1,1], dtype = float)

In [20]:
def constraint2_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return anp.array([-0.3, 0.7, -0.3], dtype = float)

In [21]:
def constraint3_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return anp.array([ot_rating_1 - 96, ot_rating_2 - 96, ot_rating_3 - 96],dtype = float)

In [22]:
def constraint4_jacobian(theta):
    x1 = theta[0]
    x2 = theta[1]
    x3 = theta[2]

    return anp.array([eff_score_1 - efficiency_score_min, eff_score_2 - efficiency_score_min, eff_score_3 - efficiency_score_min], dtype = float)

In [23]:
constraints2 = [
    {"type":"eq", "fun" : constraint1, "jac" : constraint1_jacobian},
    {"type":"ineq", "fun" : constraint2, "jac" : constraint2_jacobian},
    {"type":"ineq", "fun" : constraint3, "jac" : constraint3_jacobian},
    {"type":"ineq", "fun" : constraint4, "jac" : constraint4_jacobian}
]

In [24]:
sol2 = minimize(fun = objective, x0 = theta0, method = "SLSQP", jac = objective_gradient, bounds = bounds, constraints = constraints2)

In [25]:
sol2

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 655681.8181781151
           x: [ 2.500e+02  4.409e+02  1.091e+02]
         nit: 3
         jac: [ 6.000e+02  8.500e+02  1.200e+03]
        nfev: 3
        njev: 2
 multipliers: [ 9.773e+02  0.000e+00  6.364e+01  0.000e+00]

## Method 3: Using SciPy and Autograd to calculate constraint jacobians and objective gradient 

In [26]:
objective_grad_2 = grad(objective)

In [27]:
con1_jacobian = jacobian(constraint1)
con2_jacobian = jacobian(constraint2)
con3_jacobian = jacobian(constraint3)
con4_jacobian = jacobian(constraint4)

In [28]:
constraints3 = [
    {"type":"eq", "fun":constraint1, "jac":con1_jacobian},
    {"type":"ineq", "fun":constraint2, "jac":con2_jacobian},
    {"type":"ineq", "fun":constraint3, "jac":con3_jacobian},
    {"type":"ineq", "fun":constraint4, "jac":con4_jacobian}
]

In [29]:
sol3 = minimize(fun = objective, x0 = theta0, method = "SLSQP", jac = objective_grad_2, constraints = constraints3, bounds = bounds)

In [30]:
sol3

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 655681.8181781151
           x: [ 2.500e+02  4.409e+02  1.091e+02]
         nit: 3
         jac: [ 6.000e+02  8.500e+02  1.200e+03]
        nfev: 3
        njev: 2
 multipliers: [ 9.773e+02  0.000e+00  6.364e+01  0.000e+00]

## Method 4: Using DOcplex

In [31]:
mdl = Model(name = "Freight_Optimizer")
x1 = mdl.integer_var(lb = 0, ub = internal_fleet_load_max, name = "Internal_Fleet_Loads")
x2 = mdl.integer_var(lb = 0, name = "Dedicated_Contract_Loads")
x3 = mdl.integer_var(lb = 0, name = "Expedited_Carrier_Loads")

mdl.add_constraint(x1 + x2 + x3 - total_load_target == 0, ctname = "Total_Loads_Needed")
mdl.add_constraint(0.7*x2 - 0.3*x1 - 0.3*x3 >= 0, ctname = "Dedicated_Contract_Load_Minimum")
mdl.add_constraint(((ot_rating_1 - 96) * x1 + (ot_rating_2 - 96) * x2 + (ot_rating_3 - 96) * x3) >= 0, ctname = "On_Time_Rating_Minimum")
mdl.add_constraint(((eff_score_1 - efficiency_score_min) * x1 + (eff_score_2 - efficiency_score_min) * x2 + (eff_score_3 - efficiency_score_min) * x3) >= 0, ctname = "Efficiency_Minimum")

docplex.mp.LinearConstraint[Efficiency_Minimum](Internal_Fleet_Loads-Expedited_Carrier_Loads,GE,0)

In [32]:
mdl.minimize(cost_rate_1 * x1 + cost_rate_2 * x2 + cost_rate_3 * x3)

In [33]:
sol4 = mdl.solve()

In [34]:
print("The mimimum cost is {} dollars.".format(int(round(sol4.get_objective_value(), 0))))
print("The optimum load quantity needed from internal fleet is {}".format(round(sol4.get_value(x1),3)))
print("The optimum load quantity needed from dedicated contract carriers is {}".format(round(sol4.get_value(x2),3)))
print("The optimum load quantity needed from expedited carriers is {}".format(round(sol4.get_value(x3),3)))

The mimimum cost is 656000 dollars.
The optimum load quantity needed from internal fleet is 250.0
The optimum load quantity needed from dedicated contract carriers is 440.0
The optimum load quantity needed from expedited carriers is 110.0


## Comparison with Unoptimized Solutions

### The below code was generated by Gemini in order for me to compare what the potential cost savings were by doing this optimization. The first code block compares 2 unoptimized solutions with the optimized one while the second code block creates an interactive tool allowing the user to adjust values for x1, x2, x3

In [35]:
# ==========================================
# 1. DATA EXTRACTION (From your existing DataFrame)
# ==========================================
costs = data["Cost_Per_Load"].values
on_time_ratings = data["On_Time_Rating"].values
eff_scores = data["Efficiency_Score"].values

# ==========================================
# 2. CONFIGURATION BLOCK (Business Rules)
# ==========================================
TOTAL_VOLUME_TARGET = 800
MIN_DEDICATED_PROPORTION = 0.30
MAX_INTERNAL_CAPACITY = 250
MIN_ON_TIME_SLA = 96.0  # Fixed to 96.0 (not 0.96)
MIN_EFFICIENCY_SLA = 7.0

# Calculated dynamically
MIN_DEDICATED_VOLUME = TOTAL_VOLUME_TARGET * MIN_DEDICATED_PROPORTION


# ==========================================
# 3. SCENARIO EVALUATION TOOL
# ==========================================
def evaluate_scenario(scenario_name, allocation):
    """Calculates and prints the metrics for a given load allocation dynamically."""
    theta = np.array(allocation)
    
    # Calculate Business Metrics
    total_cost = np.dot(costs, theta)
    avg_on_time = np.dot(on_time_ratings, theta) / TOTAL_VOLUME_TARGET
    avg_eff = np.dot(eff_scores, theta) / TOTAL_VOLUME_TARGET
    
    # Check Constraints against Config Block
    vol_pass = np.sum(theta) == TOTAL_VOLUME_TARGET
    int_pass = theta[0] <= MAX_INTERNAL_CAPACITY
    ded_pass = theta[1] >= MIN_DEDICATED_VOLUME
    ot_pass = avg_on_time >= MIN_ON_TIME_SLA
    eff_pass = avg_eff >= MIN_EFFICIENCY_SLA
    
    is_valid = all([vol_pass, int_pass, ded_pass, ot_pass, eff_pass])
    
    print(f"=== {scenario_name} ===")
    print(f"Allocation [Internal, Dedicated, Expedited]: {allocation}")
    print(f"Total Cost: ${total_cost:,.0f}")
    print(f"Avg On-Time Rating: {avg_on_time:.2f}% (Pass SLA: {ot_pass})")
    print(f"Avg Fuel Efficiency: {avg_eff:.2f} (Pass SLA: {eff_pass})")
    print(f"Strictly Valid Solution: {is_valid}\n")


# ==========================================
# 4. SCIPY OPTIMIZATION MODEL
# ==========================================
def objective(theta):
    return np.dot(costs, theta)

def constraint1(theta):
    return np.sum(theta) - TOTAL_VOLUME_TARGET

def constraint2(theta):
    return theta[1] - MIN_DEDICATED_VOLUME

def constraint3(theta):
    return MAX_INTERNAL_CAPACITY - theta[0]

def constraint4(theta):
    avg_on_time = np.dot(on_time_ratings, theta) / TOTAL_VOLUME_TARGET
    return avg_on_time - MIN_ON_TIME_SLA

def constraint5(theta):
    avg_eff = np.dot(eff_scores, theta) / TOTAL_VOLUME_TARGET
    return avg_eff - MIN_EFFICIENCY_SLA

constraints = [
    {"type": "eq", "fun": constraint1},
    {"type": "ineq", "fun": constraint2},
    {"type": "ineq", "fun": constraint3},
    {"type": "ineq", "fun": constraint4},
    {"type": "ineq", "fun": constraint5}
]

bounds = [(0, None), (0, None), (0, None)]
theta0 = [250, 400, 150] # Initial guess

# Run the Optimizer
sol = minimize(fun=objective, x0=theta0, method="SLSQP", bounds=bounds, constraints=constraints)

# ==========================================
# 5. PRINT RESULTS
# ==========================================
# Evaluate human guesses
evaluate_scenario("Scenario 1: The Lazy Dispatcher", [250, 400, 150])
evaluate_scenario("Scenario 2: The Absolute Worst-Case", [250, 300, 250])

# Evaluate the optimized result
optimal_allocation = np.round(sol.x, 2)
evaluate_scenario("The Mathematical Optimum (SciPy Output)", optimal_allocation)

=== Scenario 1: The Lazy Dispatcher ===
Allocation [Internal, Dedicated, Expedited]: [250, 400, 150]
Total Cost: $670,000
Avg On-Time Rating: 96.28% (Pass SLA: True)
Avg Fuel Efficiency: 7.12 (Pass SLA: True)
Strictly Valid Solution: True

=== Scenario 2: The Absolute Worst-Case ===
Allocation [Internal, Dedicated, Expedited]: [250, 300, 250]
Total Cost: $705,000
Avg On-Time Rating: 96.97% (Pass SLA: True)
Avg Fuel Efficiency: 7.00 (Pass SLA: True)
Strictly Valid Solution: True

=== The Mathematical Optimum (SciPy Output) ===
Allocation [Internal, Dedicated, Expedited]: [250.   440.91 109.09]
Total Cost: $655,682
Avg On-Time Rating: 96.00% (Pass SLA: False)
Avg Fuel Efficiency: 7.18 (Pass SLA: True)
Strictly Valid Solution: False



In [36]:
import ipywidgets as widgets
import numpy as np
from IPython.display import HTML, display

# ==========================================
# 1. EXTRACT DATA & CONFIGURATION
# ==========================================
costs = data["Cost_Per_Load"].values
on_time_ratings = data["On_Time_Rating"].values
eff_scores = data["Efficiency_Score"].values

TOTAL_VOLUME_TARGET = 800
MIN_DEDICATED_PROPORTION = 0.30
MAX_INTERNAL_CAPACITY = 250
MIN_ON_TIME_SLA = 96.0
MIN_EFFICIENCY_SLA = 7.0
MIN_DEDICATED_VOLUME = TOTAL_VOLUME_TARGET * MIN_DEDICATED_PROPORTION

# ==========================================
# 2. INTERACTIVE UPDATE FUNCTION
# ==========================================
def update_dashboard(internal, dedicated, expedited):
    # Calculate Metrics
    total_vol = internal + dedicated + expedited
    total_cost = (
        internal * costs[0] + dedicated * costs[1] + expedited * costs[2]
    )

    # Handle division by zero if all sliders are at 0
    if total_vol > 0:
        avg_ot = (
            internal * on_time_ratings[0]
            + dedicated * on_time_ratings[1]
            + expedited * on_time_ratings[2]
        ) / total_vol
        avg_eff = (
            internal * eff_scores[0]
            + dedicated * eff_scores[1]
            + expedited * eff_scores[2]
        ) / total_vol
    else:
        avg_ot, avg_eff = 0.0, 0.0

    # Validate Constraints
    c1_ok = total_vol == TOTAL_VOLUME_TARGET
    c2_ok = internal <= MAX_INTERNAL_CAPACITY
    c3_ok = dedicated >= MIN_DEDICATED_VOLUME
    c4_ok = avg_ot >= MIN_ON_TIME_SLA
    c5_ok = avg_eff >= MIN_EFFICIENCY_SLA
    all_ok = all([c1_ok, c2_ok, c3_ok, c4_ok, c5_ok])

    # Helper function to style text based on pass/fail status
    def status_html(label, value, condition, expected_str):
        color = "green" if condition else "red"
        symbol = "✅" if condition else "❌"
        return f"<div style='margin-bottom: 5px;'><b>{label}:</b> <span style='color:{color}; font-weight:bold;'>{value}</span> {symbol} <small style='color:gray;'>({expected_str})</small></div>"

    # Construct the KPI Card Dashboard via HTML
    html_output = f"""
    <div style="font-family: Arial, sans-serif; padding: 15px; border: 1px solid #ccc; border-radius: 8px; background-color: #f9f9f9; width: 500px;">
        <h3 style="margin-top: 0; color: #333;">Live Optimization Metrics</h3>
        
        <div style="background-color: {'#e6f4ea' if all_ok else '#fce8e6'}; padding: 10px; border-radius: 4px; margin-bottom: 15px; text-align: center;">
            <b style="color: {'green' if all_ok else 'red'}; font-size: 16px;">
                {'FEASIBLE SOLUTION FOUND' if all_ok else 'INVALID SOLUTION (Fix Constraints)'}
            </b>
        </div>
        
        <div style="font-size: 20px; margin-bottom: 15px; font-weight: bold; color: #2c3e50;">
            Total Calculated Cost: ${total_cost:,.0f}
            <div style="font-size: 11px; font-weight: normal; color: #7f8c8d; margin-top: 2px;">
                *Mathematical Optimal Benchmark: $656,000
            </div>
        </div>
        
        <hr style="border: 0; border-top: 1px solid #eee; margin-bottom: 10px;">
        
        {status_html("Total Volume (Loads)", f"{total_vol} / {TOTAL_VOLUME_TARGET}", c1_ok, f"Must equal exactly {TOTAL_VOLUME_TARGET}")}
        {status_html("Internal Fleet Capacity", f"{internal} loads", c2_ok, f"Must be &le; {MAX_INTERNAL_CAPACITY}")}
        {status_html("Dedicated Minimum Volume", f"{dedicated} loads", c3_ok, f"Must be &ge; {int(MIN_DEDICATED_VOLUME)}")}
        {status_html("Blended On-Time Rating", f"{avg_ot:.2f}%", c4_ok, f"Must be &ge; {MIN_ON_TIME_SLA}%")}
        {status_html("Average Fuel Efficiency", f"{avg_eff:.2f}", c5_ok, f"Must be &ge; {MIN_EFFICIENCY_SLA}")}
    </div>
    """
    display(HTML(html_output))


# ==========================================
# 3. DEFINE INTERACTIVE SLIDERS
# ==========================================
# Initialize sliders at the original "Lazy Dispatcher" human guess state
slider_internal = widgets.IntSlider(
    value=250,
    min=0,
    max=300,
    step=5,
    description="Internal:",
    continuous_update=True,
)
slider_dedicated = widgets.IntSlider(
    value=400,
    min=0,
    max=800,
    step=5,
    description="Dedicated:",
    continuous_update=True,
)
slider_expedited = widgets.IntSlider(
    value=150,
    min=0,
    max=800,
    step=5,
    description="Expedited:",
    continuous_update=True,
)

# Render the layout using widgets.interactive
ui = widgets.VBox([slider_internal, slider_dedicated, slider_expedited])
out = widgets.interactive_output(
    update_dashboard,
    {
        "internal": slider_internal,
        "dedicated": slider_dedicated,
        "expedited": slider_expedited,
    },
)

display(ui, out)

Output()